# AmsiNet - Step 19: paired tests for the EfficientNet-B0 word agent (CPU, a few minutes)
Step 17 showed that a flat EfficientNet-B0 with free context beats the old AmsiNet cascade (EfficientViT-B0 word agent) at the same cost. This notebook decides how the paper is rebuilt around EfficientNet-B0 as the word agent. It uses the same pipeline as Steps 9 and 17 (fusion weights and tau* fitted on validation, test once) and reports:
1. paired tests (paired bootstrap over test images, averaged over the five seeds, plus per-seed intervals) for the key comparisons, above all **EfficientNet-B0 + cascade vs EfficientNet-B0 + free context** (does the cascade add anything?);
2. the cost-accuracy operating points, with a check of which systems are on the accuracy-cost frontier;
3. per-script F1 for the new system against flat ViT-B.
Run it on a CPU runtime with *Run all*, then paste back the outputs of sections 5, 6, 7 and 8.

In [ ]:
# 1. Setup
from google.colab import drive
import os
if not os.path.ismount('/content/drive'): drive.mount('/content/drive')
assert os.path.exists('/content/drive/MyDrive/amsinet/data/bstd_index.json'), 'Drive is not mounted or amsinet/data is missing: reconnect and run this cell again'
import json, numpy as np, pandas as pd
from scipy.special import softmax
from scipy import stats
ROOT = '/content/drive/MyDrive/amsinet'
CFG = dict(SEEDS=[0, 1, 2, 3, 4], VARIANTS=['word', 'word_moe'], EPS=1e-4, KEEP=0.90)
D = json.load(open(f'{ROOT}/data/bstd_index.json', encoding='utf-8'))
SCRIPTS, INDEX = D['scripts'], D['index']; SIDX = {s: i for i, s in enumerate(SCRIPTS)}; C = len(SCRIPTS); IDX = {r['id']: r for r in INDEX}
META = pd.read_csv(f'{ROOT}/data/crops_meta.csv'); REGDF = pd.read_csv(f'{ROOT}/data/regions_meta.csv')
eff = pd.read_csv(f'{ROOT}/agents/agent_efficiency_cpu.csv').set_index('agent')['GFLOPs']
flat_df = pd.read_csv(f'{ROOT}/results/backbone_efficiency.csv'); FLAT = float(flat_df[flat_df.agent == 'Flat baseline']['GFLOPs'].iloc[0])
missing = [f'{t}_seed{s}' for s in CFG['SEEDS'] for t in ('block', 'line', 'word', 'word_moe') if not os.path.exists(f'{ROOT}/agents/{t}_seed{s}_test_probs.npy')]
print('missing agent files:', missing or 'none')

In [ ]:
# 2. Helpers (same definitions as Steps 6 and 8)
def load(tag, sp):
    P = np.load(f'{ROOT}/agents/{tag}_{sp}_probs.npy').astype(np.float32); pa = pd.read_csv(f'{ROOT}/agents/{tag}_{sp}_paths.csv'); return P, pa.path.tolist()
def region_lookup(level, tag, sp):
    P, paths = load(tag, sp); pos = {p: i for i, p in enumerate(paths)}
    d = REGDF[(REGDF.level == level) & (REGDF.split == sp)]
    return {(r.image_id, r.rid): P[pos[r.path]] for r in d.itertuples() if r.path in pos}
def table(sp, wtag, ltag, btag):
    d = META[META.split == sp].reset_index(drop=True); n = len(d)
    Pw, pw = load(wtag, sp); assert pw == d.path.tolist(), 'word probabilities are not aligned with crops_meta.csv'
    logP = np.log(np.clip(Pw, CFG['EPS'], 1)); lid, bid = [], []
    for iid, k in zip(d.image_id, d.word_idx):
        w = IDX[iid]['words'][k]; lid.append(w.get('line', -1)); bid.append(w.get('block', -1))
    img = d.image_id.values; lkey = img.astype(str) + '|l' + np.array(lid).astype(str); bkey = img.astype(str) + '|b' + np.array(bid).astype(str)
    lg, lu = pd.factorize(lkey); bg, bu = pd.factorize(bkey)
    def loo(g, ng):
        S = np.zeros((ng, C)); cnt = np.zeros(ng); np.add.at(S, g, logP); np.add.at(cnt, g, 1); o = cnt[g] - 1
        F = np.zeros((n, C)); m = o > 0; F[m] = (S[g][m] - logP[m]) / o[m][:, None]; return F
    def reg(level, tag, key):
        L = region_lookup(level, tag, sp); F = np.zeros((n, C))
        for i, (iid, kk) in enumerate(zip(img, key)):
            v = L.get((iid, f'{level[0]}{kk}'))
            if v is not None: F[i] = np.log(np.clip(v, CFG['EPS'], 1))
        return F
    ui, inv = np.unique(img, return_inverse=True); l_img = np.zeros(len(lu), int); l_img[lg] = inv; b_img = np.zeros(len(bu), int); b_img[bg] = inv
    return dict(y=np.array([SIDX[s] for s in d.script]), word=logP, line_ctx=loo(lg, len(lu)), block_ctx=loo(bg, len(bu)),
                line_agent=reg('line', ltag, lid), block_agent=reg('block', btag, bid),
                img=img, inv=inv, n_img=len(ui), lg=lg, bg=bg, nl=len(lu), nb=len(bu), l_img=l_img, b_img=b_img)
def confmat(y, p): return np.bincount(y * C + p, minlength=C * C).reshape(C, C)
def f1_from(cm):
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; den = 2 * tp + fp + fn; return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
def metr(y, p): cm = confmat(y, p); return dict(acc=100 * float(np.trace(cm) / cm.sum()), f1=100 * float(f1_from(cm).mean()))
def unc(s): p = softmax(s, axis=1); return -(p * np.log(np.clip(p, 1e-12, 1))).sum(1) / np.log(C)
KEYS = ['line_ctx', 'block_ctx', 'line_agent', 'block_agent']; GRID_W = [0, .1, .25, .5, 1, 1.5, 2, 3]
def cascade(T, tau, W):
    s = T['word'] + W['line_ctx'] * T['line_ctx'] + W['block_ctx'] * T['block_ctx']; u = unc(s)
    li = np.zeros(T['nl'], bool); np.logical_or.at(li, T['lg'], u > tau); s = s + W['line_agent'] * T['line_agent'] * li[T['lg']][:, None]
    u = unc(s); bi = np.zeros(T['nb'], bool); np.logical_or.at(bi, T['bg'], u > tau); s = s + W['block_agent'] * T['block_agent'] * bi[T['bg']][:, None]
    return s, li, bi
def cost(T, li, bi, GF):
    nw = np.bincount(T['inv'], minlength=T['n_img']).astype(float); nl = np.bincount(T['l_img'][li], minlength=T['n_img']).astype(float); nb = np.bincount(T['b_img'][bi], minlength=T['n_img']).astype(float)
    return float((nw * GF['word'] + nl * GF['line'] + nb * GF['block']).mean())
TAUS = [0, .02, .05, .1, .15, .2, .3, .4, .5, .6, .7, .8, .9, 1.01]

## 3. Costs per word and the list of flat models that are available

In [ ]:
try:
    import torch
except ImportError: torch = None
GFW = {'word': float(eff['word']), 'word_moe': float(eff['word_moe']), 'flatvit': FLAT}
TAG = {'word': 'word', 'word_moe': 'word_moe', 'flatvit': 'flatvit'}
CAND = {'flat_vits': ('vit_small_patch16_224.augreg_in21k', 9.2), 'flat_effb0': ('efficientnet_b0.ra_in1k', 0.8)}   # fallback GFLOPs if the counter is unavailable
for name, (mn, fb) in CAND.items():
    if all(os.path.exists(f'{ROOT}/agents/{name}_seed{s}_{sp}_probs.npy') for s in CFG['SEEDS'] for sp in ('val', 'test')):
        try:
            import timm
            from torch.utils.flop_counter import FlopCounterMode
            m = timm.create_model(mn, pretrained=False, num_classes=C).eval(); x = torch.zeros(1, 3, 224, 224)
            with torch.no_grad(), FlopCounterMode(display=False) as fc: m(x)
            gf = fc.get_total_flops() / 1e9
        except Exception as e: print('FLOP counter failed for', name, e, '-> using', fb); gf = fb
        GFW[name] = gf; TAG[name] = name; print(f'{name}: {gf:.2f} GFLOPs per word')
    else: print('not available yet:', name)
VARS = [v for v in ['word_moe', 'word', 'flatvit', 'flat_vits', 'flat_effb0'] if v in GFW]; print('variants:', VARS)

## 4. One run per seed and model

In [ ]:
def run(seed, variant, keep_preds=False):
    wtag, ltag, btag = f'{TAG[variant]}_seed{seed}', f'line_seed{seed}', f'block_seed{seed}'
    GF = dict(word=GFW[variant], line=float(eff['line']), block=float(eff['block']))
    TV, TT = table('val', wtag, ltag, btag), table('test', wtag, ltag, btag)
    def full(T, W): return T['word'] + sum(W[k] * T[k] for k in KEYS)
    vf1 = lambda W: metr(TV['y'], full(TV, W).argmax(1))['f1']
    W = {k: 0.0 for k in KEYS}
    for _ in range(2):
        for k in KEYS: W[k] = max(GRID_W, key=lambda g: vf1({**W, k: g}))
    cv = {t: (lambda r: (metr(TV['y'], r[0].argmax(1))['f1'], cost(TV, r[1], r[2], GF)))(cascade(TV, t, W)) for t in TAUS}
    f_full, f_ctx = cv[0][0], cv[1.01][0]; target = f_ctx + CFG['KEEP'] * (f_full - f_ctx)
    ok = [t for t in TAUS if cv[t][0] >= target]; tau = min(ok, key=lambda t: cv[t][1]) if ok else 0.0
    out = dict(seed=seed, variant=variant, tau_star=tau, **{f'w_{k}': W[k] for k in KEYS})
    y = TT['y']; res = {}
    for name, (s, li, bi) in dict(word=(TT['word'], np.zeros(TT['nl'], bool), np.zeros(TT['nb'], bool)), context=cascade(TT, 1.01, W), cascade=cascade(TT, tau, W), full=cascade(TT, 0, W)).items():
        m = metr(y, s.argmax(1)); out[f'{name}_acc'], out[f'{name}_f1'], out[f'{name}_gflops'] = m['acc'], m['f1'], cost(TT, li, bi, GF); res[name] = s.argmax(1)
        if name == 'cascade': out['lines_pct'], out['blocks_pct'] = 100 * li.mean(), 100 * bi.mean()
    # mixed-script lines: lines whose words have more than one true script
    first = np.full(TT['nl'], -1); mixed = np.zeros(TT['nl'], bool)
    for g, yy in zip(TT['lg'], y):
        if first[g] < 0: first[g] = yy
        elif first[g] != yy: mixed[g] = True
    mw = mixed[TT['lg']]; out['mixed_line_pct'] = 100 * mixed.mean(); out['mixed_word_pct'] = 100 * mw.mean()
    for name in ('word', 'context', 'cascade', 'full'):
        out[f'{name}_acc_mixed'] = 100 * float((res[name][mw] == y[mw]).mean()); out[f'{name}_acc_single'] = 100 * float((res[name][~mw] == y[~mw]).mean())
    return (out, dict(y=y, **res, mw=mw, conf=np.exp(TT['word']).max(1), nline=np.bincount(TT['lg'])[TT['lg']], paths=META[META.split == 'test'].path.tolist(), img=TT['img'])) if keep_preds else (out, None)

## 4. One run per seed for the systems that are compared (keeps the test predictions)

In [ ]:
NEED = [v for v in ['flat_effb0', 'word_moe', 'flatvit', 'flat_vits'] if v in GFW]
assert 'flat_effb0' in NEED and 'word_moe' in NEED and 'flatvit' in NEED, 'flat_effb0 (Step 16) is not on Drive: run Step 16 first'
rows, PRED = [], {}
for v in NEED:
    for seed in CFG['SEEDS']:
        o, p = run(seed, v, keep_preds=True); rows.append(o); PRED[(v, seed)] = p
        print(v, seed, 'tau*', o['tau_star'], '| cascade %.2f/%.2f (%.1f GF)' % (o['cascade_acc'], o['cascade_f1'], o['cascade_gflops']))
PS = pd.DataFrame(rows); PS.to_csv(f'{ROOT}/results/step19_per_seed.csv', index=False)

## 5. Operating points (mean +/- 95% CI over seeds)  -  paste this back

In [ ]:
def mci(x):
    x = np.asarray(x, float); n = len(x); return x.mean(), (stats.t.ppf(.975, n - 1) * x.std(ddof=1) / np.sqrt(n) if n > 1 else float('nan'))
NAMES = {'word_moe': 'AmsiNet (EfficientViT-B0 MoE)', 'flatvit': 'flat ViT-B/16', 'flat_vits': 'flat ViT-S/16', 'flat_effb0': 'EfficientNet-B0'}
LAB = {'word': 'alone', 'context': '+ context', 'cascade': '+ cascade'}
OP = []
for v in NEED:
    d = PS[PS.variant == v]
    for k in ('word', 'context', 'cascade'):
        OP.append(dict(system=f'{NAMES[v]} {LAB[k]}', v=v, k=k, acc=d[f'{k}_acc'].mean(), acc_ci=mci(d[f'{k}_acc'])[1], f1=d[f'{k}_f1'].mean(), f1_ci=mci(d[f'{k}_f1'])[1], gflops=d[f'{k}_gflops'].mean(),
                       tau=d.tau_star.mean() if k == 'cascade' else np.nan, lines=d.lines_pct.mean() if k == 'cascade' else np.nan, blocks=d.blocks_pct.mean() if k == 'cascade' else np.nan))
OPD = pd.DataFrame(OP)
OPD['on_frontier'] = [not any((o2.acc >= o.acc and o2.f1 >= o.f1 and o2.gflops <= o.gflops) and (o2.acc > o.acc or o2.f1 > o.f1 or o2.gflops < o.gflops) for o2 in OPD.itertuples()) for o in OPD.itertuples()]
show = OPD.copy(); show['accuracy'] = show.apply(lambda r: f'{r.acc:.2f} +/- {r.acc_ci:.2f}', axis=1); show['macro-F1'] = show.apply(lambda r: f'{r.f1:.2f} +/- {r.f1_ci:.2f}', axis=1)
show['GFLOPs/img'] = show.gflops.round(1); show['tau*'] = show.tau.round(2); show['lines %'] = show.lines.round(1); show['blocks %'] = show.blocks.round(1)
display(show[['system', 'accuracy', 'macro-F1', 'GFLOPs/img', 'tau*', 'lines %', 'blocks %', 'on_frontier']])
OPD.to_csv(f'{ROOT}/results/step19_operating_points.csv', index=False)

## 6. Paired comparisons  -  paste this back
Each line: difference in test accuracy and macro-F1 (A minus B), mean over seeds. Bracket = 95% paired bootstrap over test images (2000 resamples, same resample for all seeds); +/- = 95% t-interval over the five seeds.

In [ ]:
def img_cm(v, k, seed):
    p = PRED[(v, seed)]; inv = pd.factorize(p['img'])[0]; n_img = inv.max() + 1
    M = np.zeros((n_img, C * C)); np.add.at(M, (inv, p['y'] * C + p[k]), 1); return M
CM = {(v, k, s): img_cm(v, k, s) for v in NEED for k in ('word', 'context', 'cascade') for s in CFG['SEEDS']}
n_img = next(iter(CM.values())).shape[0]; rng = np.random.RandomState(0); Bn = 2000
Wb = rng.multinomial(n_img, np.ones(n_img) / n_img, size=Bn).astype(float)
def boot(v, k):
    acc = np.zeros(Bn); f1 = np.zeros(Bn)
    for s in CFG['SEEDS']:
        cnt = Wb @ CM[(v, k, s)]; cm = cnt.reshape(Bn, C, C); tp = np.einsum('bii->bi', cm); fp = cm.sum(1) - tp; fn = cm.sum(2) - tp
        den = 2 * tp + fp + fn; f = np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
        acc += 100 * tp.sum(1) / cm.sum((1, 2)); f1 += 100 * f.mean(1)
    return acc / len(CFG['SEEDS']), f1 / len(CFG['SEEDS'])
BS = {(v, k): boot(v, k) for v in NEED for k in ('word', 'context', 'cascade')}
def pair(a, b, text):
    (va, ka), (vb, kb) = a, b; da = PS[PS.variant == va].set_index('seed'); db = PS[PS.variant == vb].set_index('seed')
    sa = (da[f'{ka}_acc'] - db[f'{kb}_acc']).values; sf = (da[f'{ka}_f1'] - db[f'{kb}_f1']).values
    ba = BS[a][0] - BS[b][0]; bf = BS[a][1] - BS[b][1]; q = lambda x: np.percentile(x, [2.5, 97.5])
    print(f"{text}: accuracy {sa.mean():+.2f} [{q(ba)[0]:+.2f}, {q(ba)[1]:+.2f}] (+/- {mci(sa)[1]:.2f}) | macro-F1 {sf.mean():+.2f} [{q(bf)[0]:+.2f}, {q(bf)[1]:+.2f}] (+/- {mci(sf)[1]:.2f}) | GFLOPs {da[f'{ka}_gflops'].mean():.1f} vs {db[f'{kb}_gflops'].mean():.1f}")
E, M_, V, S = 'flat_effb0', 'word_moe', 'flatvit', 'flat_vits'
pair((E, 'cascade'), (E, 'context'), 'EffNet-B0 cascade minus EffNet-B0 context (does the cascade help?)')
pair((E, 'context'), (E, 'word'), 'EffNet-B0 context minus EffNet-B0 alone (value of context)')
pair((E, 'cascade'), (M_, 'cascade'), 'EffNet-B0 cascade minus old AmsiNet cascade')
pair((E, 'context'), (M_, 'cascade'), 'EffNet-B0 context minus old AmsiNet cascade')
pair((E, 'cascade'), (V, 'word'), 'EffNet-B0 cascade minus flat ViT-B alone')
pair((E, 'cascade'), (V, 'context'), 'EffNet-B0 cascade minus flat ViT-B context')
pair((E, 'cascade'), (V, 'cascade'), 'EffNet-B0 cascade minus flat ViT-B cascade')
pair((E, 'context'), (V, 'context'), 'EffNet-B0 context minus flat ViT-B context')
if S in NEED:
    pair((E, 'cascade'), (S, 'cascade'), 'EffNet-B0 cascade minus flat ViT-S cascade')
    pair((E, 'cascade'), (S, 'context'), 'EffNet-B0 cascade minus flat ViT-S context')

## 7. Per-script F1 (mean over seeds)  -  paste this back

In [ ]:
def f1s(v, k):
    out = []
    for s in CFG['SEEDS']:
        cm = CM[(v, k, s)].sum(0).reshape(C, C); out.append(100 * f1_from(cm.astype(int)))
    return np.mean(out, 0)
y_all = PRED[(E, 0)]['y']; cnt = np.bincount(y_all, minlength=C)
PSC = pd.DataFrame({'script': SCRIPTS, 'test words': cnt,
                    'EffNet-B0 alone': f1s(E, 'word'), 'EffNet-B0 + context': f1s(E, 'context'), 'EffNet-B0 + cascade': f1s(E, 'cascade'),
                    'flat ViT-B alone': f1s(V, 'word'), 'flat ViT-B + context': f1s(V, 'context'), 'old AmsiNet cascade': f1s(M_, 'cascade')})
PSC['cascade - ViT-B alone'] = PSC['EffNet-B0 + cascade'] - PSC['flat ViT-B alone']
PSC['cascade - ViT-B context'] = PSC['EffNet-B0 + cascade'] - PSC['flat ViT-B + context']
PSC = PSC.sort_values('test words'); display(PSC.round(1)); PSC.to_csv(f'{ROOT}/results/step19_per_script.csv', index=False)

## 8. Mixed-script lines (does context hurt when a line contains two scripts?)  -  paste this back

In [ ]:
d = PS[PS.variant == E]
print('share of test lines that are mixed-script: %.1f%%, share of words in them: %.1f%%' % (d.mixed_line_pct.mean(), d.mixed_word_pct.mean()))
for k in ('word', 'context', 'cascade'):
    print(f'EffNet-B0 {LAB[k]:10s} accuracy on words in single-script lines %.2f | in mixed-script lines %.2f' % (d[f'{k}_acc_single'].mean(), d[f'{k}_acc_mixed'].mean()))